# 01 — Подготовка данных

Источник: `DeepPavlov/massive_ru` — 60 интент-классов, русский язык.

**Синтетика не используется.** Только реальные данные → честные метрики.

Из 60 MASSIVE-интентов маппим в **5 call-center классов** строго по смыслу.
Классы `fraud_report` и `escalation_request` исключены — в MASSIVE нет аналогов.

| Класс | Описание | MASSIVE-источники |
|---|---|---|
| `technical_issue` | Проблемы с устройствами/сервисом | все `iot_*` |
| `account_management` | Управление аккаунтом, настройки | `email_*`, `social_*`, `lists_*`, `calendar_*`, `alarm_*` |
| `general_inquiry` | Справочные вопросы | `qa_*`, `news_*`, `weather_*`, `transport_*`, `recommendation_*`, `datetime_*`, `general_*`, `takeaway_query` |
| `complaint` | Негативная оценка, недовольство | `music_dislikeness` |
| `entertainment` | Развлечения, медиа, еда | `play_*`, `music_query`, `music_likeness`, `music_settings`, `cooking_recipe`, `takeaway_order` |

Выход: `data/train.csv`, `data/val.csv`, `data/test.csv`, `data/label2id.json`, `data/id2label.json`

In [ ]:
!pip install datasets pandas scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import os, json

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

## 1. Загрузка MASSIVE RU

In [ ]:
dataset = load_dataset('DeepPavlov/massive_ru')
train_raw = dataset['train'].to_pandas()
val_raw   = dataset['validation'].to_pandas()
test_raw  = dataset['test'].to_pandas()

print('Train:', len(train_raw))
print('Columns:', train_raw.columns.tolist())
print('Пример intent:', train_raw['intent'].iloc[0])
print('\nВсе интенты в датасете:')
print(sorted(train_raw['intent'].unique()))

## 2. Маппинг MASSIVE → 5 call-center классов

In [ ]:
# Строго по фактическому списку интентов датасета — без выдумок
MASSIVE_TO_CALLCENTER = {
    # technical_issue — проблемы с устройствами / техническими сервисами
    'iot_cleaning':        'technical_issue',
    'iot_coffee':          'technical_issue',
    'iot_hue_lightchange': 'technical_issue',
    'iot_hue_lightdim':    'technical_issue',
    'iot_hue_lightoff':    'technical_issue',
    'iot_hue_lighton':     'technical_issue',
    'iot_hue_lightup':     'technical_issue',
    'iot_wemo_off':        'technical_issue',
    'iot_wemo_on':         'technical_issue',

    # account_management — управление аккаунтом, расписание, списки
    'alarm_query':         'account_management',
    'alarm_remove':        'account_management',
    'alarm_set':           'account_management',
    'calendar_query':      'account_management',
    'calendar_remove':     'account_management',
    'calendar_set':        'account_management',
    'email_addcontact':    'account_management',
    'email_query':         'account_management',
    'email_querycontact':  'account_management',
    'email_sendemail':     'account_management',
    'lists_createoradd':   'account_management',
    'lists_query':         'account_management',
    'lists_remove':        'account_management',
    'social_post':         'account_management',
    'social_query':        'account_management',

    # general_inquiry — справочные / информационные запросы
    'datetime_convert':         'general_inquiry',
    'datetime_query':           'general_inquiry',
    'general_greet':            'general_inquiry',
    'general_joke':             'general_inquiry',
    'general_quirky':           'general_inquiry',
    'news_query':               'general_inquiry',
    'qa_currency':              'general_inquiry',
    'qa_definition':            'general_inquiry',
    'qa_factoid':               'general_inquiry',
    'qa_maths':                 'general_inquiry',
    'qa_stock':                 'general_inquiry',
    'recommendation_events':    'general_inquiry',
    'recommendation_locations': 'general_inquiry',
    'recommendation_movies':    'general_inquiry',
    'takeaway_query':           'general_inquiry',
    'transport_query':          'general_inquiry',
    'transport_taxi':           'general_inquiry',
    'transport_ticket':         'general_inquiry',
    'transport_traffic':        'general_inquiry',
    'weather_query':            'general_inquiry',

    # complaint — негативная оценка / недовольство
    'music_dislikeness':  'complaint',

    # entertainment — медиа, музыка, игры, еда
    'cooking_recipe':   'entertainment',
    'music_likeness':   'entertainment',
    'music_query':      'entertainment',
    'music_settings':   'entertainment',
    'play_audiobook':   'entertainment',
    'play_game':        'entertainment',
    'play_music':       'entertainment',
    'play_podcasts':    'entertainment',
    'play_radio':       'entertainment',
    'takeaway_order':   'entertainment',
    'audio_volume_down': 'entertainment',
    'audio_volume_mute': 'entertainment',
    'audio_volume_up':   'entertainment',
}

# Проверяем покрытие
all_intents = set(train_raw['intent'].unique())
mapped_intents = set(MASSIVE_TO_CALLCENTER.keys())
unmapped = all_intents - mapped_intents
print(f'Всего интентов в датасете: {len(all_intents)}')
print(f'Замаппено: {len(mapped_intents)}')
print(f'Не замаппено (отброшено): {len(unmapped)}')
print('Отброшенные:', sorted(unmapped))

In [ ]:
for df in [train_raw, val_raw, test_raw]:
    df['callcenter_intent'] = df['intent'].map(MASSIVE_TO_CALLCENTER)

mapped_train = train_raw.dropna(subset=['callcenter_intent'])
mapped_val   = val_raw.dropna(subset=['callcenter_intent'])
mapped_test  = test_raw.dropna(subset=['callcenter_intent'])

print('После маппинга:')
print(f'  Train: {len(mapped_train)} / {len(train_raw)}')
print(f'  Val:   {len(mapped_val)} / {len(val_raw)}')
print(f'  Test:  {len(mapped_test)} / {len(test_raw)}')
print('\nРаспределение классов (train):')
print(mapped_train['callcenter_intent'].value_counts())

## 3. Предупреждение о классе complaint

`complaint` в MASSIVE покрыт только `music_dislikeness` — это слабое соответствие.
Проверяем сколько примеров получается. Если < 100 — объединяем с `general_inquiry`
или убираем класс совсем.

In [ ]:
complaint_count = (mapped_train['callcenter_intent'] == 'complaint').sum()
print(f'Примеров complaint в train: {complaint_count}')

MIN_CLASS_SIZE = 100  # минимум для надёжного обучения

if complaint_count < MIN_CLASS_SIZE:
    print(f'⚠️  Мало примеров (< {MIN_CLASS_SIZE}). Объединяем complaint → general_inquiry.')
    for df in [mapped_train, mapped_val, mapped_test]:
        df.loc[df['callcenter_intent'] == 'complaint', 'callcenter_intent'] = 'general_inquiry'
    FINAL_CLASSES = ['technical_issue', 'account_management', 'general_inquiry', 'entertainment']
    print('Итоговые классы (4):', FINAL_CLASSES)
else:
    print(f'✓  Достаточно примеров. Оставляем complaint как отдельный класс.')
    FINAL_CLASSES = ['technical_issue', 'account_management', 'general_inquiry', 'complaint', 'entertainment']
    print('Итоговые классы (5):', FINAL_CLASSES)

print('\nФинальное распределение:')
print(mapped_train['callcenter_intent'].value_counts())

## 4. Числовые метки

In [ ]:
LABEL2ID = {cls: idx for idx, cls in enumerate(sorted(FINAL_CLASSES))}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

print(f'Классов: {NUM_LABELS}')
for k, v in LABEL2ID.items():
    print(f'  {v}: {k}')

for df in [mapped_train, mapped_val, mapped_test]:
    df['label'] = df['callcenter_intent'].map(LABEL2ID)

assert mapped_train['label'].isna().sum() == 0, 'Есть NaN в метках!'
mapped_train['label'] = mapped_train['label'].astype(int)
mapped_val['label']   = mapped_val['label'].astype(int)
mapped_test['label']  = mapped_test['label'].astype(int)
print('✓ Метки проставлены')

## 5. Сохранение артефактов

Используем оригинальные сплиты MASSIVE (train/val/test) — они уже стратифицированы.

In [ ]:
cols = ['utt', 'label', 'callcenter_intent']

train_out = mapped_train[cols].rename(columns={'utt': 'text', 'callcenter_intent': 'intent'})
val_out   = mapped_val[cols].rename(columns={'utt': 'text', 'callcenter_intent': 'intent'})
test_out  = mapped_test[cols].rename(columns={'utt': 'text', 'callcenter_intent': 'intent'})

# Перемешиваем train
train_out = train_out.sample(frac=1, random_state=42).reset_index(drop=True)

train_out.to_csv('data/train.csv', index=False)
val_out.to_csv('data/val.csv',     index=False)
test_out.to_csv('data/test.csv',   index=False)

with open('data/label2id.json', 'w', encoding='utf-8') as f:
    json.dump(LABEL2ID, f, ensure_ascii=False, indent=2)
with open('data/id2label.json', 'w', encoding='utf-8') as f:
    json.dump({str(k): v for k, v in ID2LABEL.items()}, f, ensure_ascii=False, indent=2)

# Сохраняем NUM_LABELS для ноутбуков 02 и 03
with open('data/dataset_info.json', 'w', encoding='utf-8') as f:
    json.dump({
        'num_labels': NUM_LABELS,
        'classes': FINAL_CLASSES,
        'label2id': LABEL2ID,
        'source': 'DeepPavlov/massive_ru',
        'synthetic': False,
        'n_train': len(train_out),
        'n_val':   len(val_out),
        'n_test':  len(test_out),
    }, f, ensure_ascii=False, indent=2)

print('Сохранено в data/')
print(f'  train.csv: {len(train_out)}')
print(f'  val.csv:   {len(val_out)}')
print(f'  test.csv:  {len(test_out)}')
print(f'  label2id.json, id2label.json, dataset_info.json')

## 6. Базовая статистика

In [ ]:
print('Длина текстов в train (символы):')
print(train_out['text'].str.len().describe().round(1))

print(f'\nПо одному примеру из каждого класса ({NUM_LABELS} классов):')
for label_id, label_name in ID2LABEL.items():
    subset = train_out[train_out['label'] == label_id]
    example = subset['text'].iloc[0]
    print(f'  [{label_name}] {example[:80]}')